
# AI 301 Final Project: Movie Memory

**Group:** 1

**Team Members:** Arianna Rogers, Drew Woodard, Dylan Welby, and Lauren Willis

**Dataset:** Wiki Movie Plots with Summaries

**Project Goal:** Identify movies from vague plot descriptions using semantic search and retrieval-augmented generation (RAG), and compare the results against a BM25 keyword search baseline.


### Project Evaluation Plan

We will evaluate Movie Memory using two baselines:

- **Search Baseline:** BM25 keyword search will be compared with semantic search using Hit Rate@5.
- **RAG Baseline:** Qwen without retrieved movie plots will be compared with Qwen using retrieved plots. We will evaluate whether the movie identification is correct and whether the explanation is supported by the movie information.

### Design Choice

We will compare retrieving the top 3 movie plots versus the top 5 movie plots. Both versions will use the same dataset, embedding model, language model, and test descriptions. The only difference will be the number of plots retrieved.

This will help us determine whether giving Qwen more movie information improves the results.


### Checkpoint 1 Progress Note

We have loaded and cleaned our dataset of 34,886 movies, created 20 test descriptions with expected answers and Record IDs, and evaluated our BM25 baseline, which achieved a Hit Rate@5 of 16.67%. We also tested Qwen without retrieved movie plots, which correctly identified 3 of 18 answerable movies.

Since Checkpoint 1, we implemented FAISS semantic search and connected it to Qwen to create our RAG system. Semantic search achieved a Hit Rate@5 of 38.89%. We also completed our controlled experiment comparing the top 3 and top 5 retrieved plots. Both versions correctly identified 6 of 18 answerable movies (33.33%), but neither correctly handled the two unanswerable descriptions.

We do not have any major technical blockers. However, our results show that Qwen sometimes selects the wrong movie even when semantic search retrieves the correct one. We will discuss this limitation in our final report.

## Team Contributions

- **Arianna Rogers — Data and Test Set Lead:** Dataset loading,
  cleaning, documentation, record ID assignment, test-set
  verification, retrieval verification, and failure analysis.

- **Lauren Willis — Retrieval or Model Lead:** Test-set
  finalization, BM25 keyword search, sentence embeddings,
  FAISS semantic search, Top-3 versus Top-5 design experiment,
  and retrieval code documentation.

- **Drew Woodard — Generation or Agent Lead:** Model feasibility
  testing, Qwen-only baseline, RAG pipeline implementation,
  semantic search and RAG integration, prompt development,
  and system limitations.

- **Dylan Welby — Evaluation and Demo Lead:** BM25 evaluation,
  Hit Rate@5 calculations, full-system evaluation, performance
  comparisons, final evaluation results, and results verification.

In [1]:
# Download the shared movie dataset from Google Drive
!pip install -q gdown
import gdown

file_id = "1GNBx6j23REDS9Cw94sx1sYx1zP663Juc"
dataset_path = "/content/wiki_movie_plots_deduped_with_summaries.csv"

gdown.download(id=file_id, output=dataset_path, quiet=False)

Downloading...
From: https://drive.google.com/uc?id=1GNBx6j23REDS9Cw94sx1sYx1zP663Juc
To: /content/wiki_movie_plots_deduped_with_summaries.csv
100%|██████████| 94.6M/94.6M [00:00<00:00, 125MB/s]


'/content/wiki_movie_plots_deduped_with_summaries.csv'

## Step 1: Data Exploration and Cleaning

We used the Wiki Movie Plots with Summaries dataset from Hugging Face, which contains 34,886 movies. We loaded the dataset using pandas and checked its columns, missing values, and duplicate rows.

There were no duplicate rows or missing plot summaries, so no movies needed to be removed. The Cast column had 1,422 missing values, but we did not need that column for our search system. We also assigned Record IDs based on the original spreadsheet row numbers and verified that they matched the movies in our sprint plan.

We will use the PlotSummary column for both BM25 and semantic search.

In [2]:
# Import pandas to work with the dataset
import pandas as pd

# Load the movie dataset
df = pd.read_csv(dataset_path)

# Show the dataset size and columns
print("Dataset Shape:", df.shape)
print("\nColumn Names:")
print(df.columns.tolist())

# Check for missing values
print("\nMissing Values:")
print(df.isnull().sum())

# Check for duplicate rows
print("\nDuplicate Rows:", df.duplicated().sum())

# Remove duplicate rows if there are any
df = df.drop_duplicates().copy()

# Remove movies without a plot summary
df = df.dropna(subset=['PlotSummary']).copy()

# Assign record IDs based on the original spreadsheet row numbers
df['Record ID'] = df.index + 2

# Display the first five movies
df[['Record ID', 'Title', 'Release Year', 'PlotSummary']].head()

Dataset Shape: (34886, 9)

Column Names:
['Release Year', 'Title', 'Origin/Ethnicity', 'Director', 'Cast', 'Genre', 'Wiki Page', 'Plot', 'PlotSummary']

Missing Values:
Release Year           0
Title                  0
Origin/Ethnicity       0
Director               0
Cast                1422
Genre                  0
Wiki Page              0
Plot                   0
PlotSummary            0
dtype: int64

Duplicate Rows: 0


,Record ID,Title,Release Year,PlotSummary
0,2,Kansas Saloon Smashers,1901,Carrie Nation and her followers burst into a s...
1,3,Love by the Light of the Moon,1901,"The moon, painted with a smiling face hangs ov..."
2,4,The Martyred Presidents,1901,"The film, just over a minute long, is composed..."
3,5,"Terrible Teddy, the Grizzly King",1901,The first shot is set in a wood during winter ...
4,6,Jack and the Beanstalk,1902,The earliest known adaptation of the classic f...


## Step 2: Create the Test Set

We created 20 vague movie descriptions to test how well our system can identify movies. Each answerable description includes the expected movie title, release year, Record ID, and Wikipedia source.

The test set contains 18 answerable descriptions and 2 unanswerable descriptions. The unanswerable descriptions do not provide enough information to identify a movie.

We saved the completed test set as a CSV file so we can use the same descriptions when comparing BM25 and semantic search.

In [3]:
# Load the additional test cases from GitHub
test_path = "https://raw.githubusercontent.com/aroger78/Movie-Memory/main/movie_memory_additional_15_test_cases.csv"
additional_tests = pd.read_csv(test_path)

# Create the five original test cases from our sprint plan
original_tests = pd.DataFrame([
    [1, "The one where the boy sells his cow for some beans.", "Jack and the Beanstalk", 1902, 6, "https://en.wikipedia.org/wiki/Jack_and_the_Beanstalk_(1902_film)"],
    [2, "The one where a girl follows a white rabbit, drinks something that makes her smaller, and tries to get through a tiny door.", "Alice in Wonderland", 1903, 7, "https://en.wikipedia.org/wiki/Alice_in_Wonderland_(1903_film)"],
    [3, "The one where a man slowly discovers that his life is being filmed for a secret television show.", "The Truman Show", 1998, 13380, "https://en.wikipedia.org/wiki/The_Truman_Show"],
    [4, "The one where a guy keeps getting confused about what's real, and later falls for a girl but can't always tell if he's talking to the right one.", "The Playhouse", 1921, 406, "https://en.wikipedia.org/wiki/The_Playhouse_(film)"],
    [5, "The movie I watched at the movie theater last year with my friend.", "Unanswerable", None, None, "N/A — no viewing history"]
], columns=additional_tests.columns)

# Combine the original five with the additional 15 test cases
test_df = pd.concat([original_tests, additional_tests], ignore_index=True)

# Replace six test cases with movies from different decades and genres
replacements = [
    [6, "A group of college friends visits a casino, and one becomes determined to pull off a robbery while another struggles with trauma from the war.", "5 Against the House", 1955, 6231],
    [7, "Two Americans get stranded in Egypt and become involved in a dangerous situation involving an ancient tomb and people trying to steal a preserved body.", "Abbott and Costello Meet the Mummy", 1955, 6233],
    [8, "A military officer wakes up believing several years have passed since the war, but the people around him are secretly trying to get information about an upcoming invasion.", "36 Hours", 1965, 7943],
    [10, "A famous artist is asked by a religious leader to decorate a large ceiling but initially refuses because he dislikes the proposed design.", "The Agony and the Ecstasy", 1965, 7944],
    [11, "A prisoner with computer skills is offered a chance at freedom if he helps investigators track down someone responsible for dangerous digital attacks.", "Blackhat", 2015, 16915],
    [13, "After a tragedy at a coal mine, a teenager disappears from a small town, connecting several people who are hiding secrets.", "Little Accidents", 2015, 16916]
]

# Update the descriptions, expected movies, years, record IDs, and sources
for number, description, title, year, record_id in replacements:
    row = test_df['Test'] == number
    movie = df[df['Record ID'] == record_id].iloc[0]
    test_df.loc[row, ['Description', 'Expected_Title', 'Year', 'Record_ID', 'Source']] = [
        description, title, year, record_id, movie['Wiki Page']
    ]

# Display the complete test set
test_df


,Test,Description,Expected_Title,Year,Record_ID,Source
0,1,The one where the boy sells his cow for some b...,Jack and the Beanstalk,1902.0,6.0,https://en.wikipedia.org/wiki/Jack_and_the_Bea...
1,2,"The one where a girl follows a white rabbit, d...",Alice in Wonderland,1903.0,7.0,https://en.wikipedia.org/wiki/Alice_in_Wonderl...
2,3,The one where a man slowly discovers that his ...,The Truman Show,1998.0,13380.0,https://en.wikipedia.org/wiki/The_Truman_Show
3,4,The one where a guy keeps getting confused abo...,The Playhouse,1921.0,406.0,https://en.wikipedia.org/wiki/The_Playhouse_(f...
4,5,The movie I watched at the movie theater last ...,Unanswerable,NaN,NaN,N/A — no viewing history
5,6,"A group of college friends visits a casino, an...",5 Against the House,1955.0,6231.0,https://en.wikipedia.org/wiki/5_Against_the_House
6,7,Two Americans get stranded in Egypt and become...,Abbott and Costello Meet the Mummy,1955.0,6233.0,https://en.wikipedia.org/wiki/Abbott_and_Coste...
7,8,A military officer wakes up believing several ...,36 Hours,1965.0,7943.0,https://en.wikipedia.org/wiki/36_Hours_(1965_f...
8,9,A computer hacker learns that the world around...,The Matrix,1999.0,13511.0,https://en.wikipedia.org/wiki/The_Matrix
9,10,A famous artist is asked by a religious leader...,The Agony and the Ecstasy,1965.0,7944.0,https://en.wikipedia.org/wiki/The_Agony_and_th...


In [4]:
# Set the location for the completed test set
test_output_path = '/content/movie_memory_20_test_cases.csv'
# Save all 20 test cases as a CSV file
test_df.to_csv(test_output_path, index=False)

# Confirm the test set was saved
print("Saved", len(test_df), "test cases to Colab")

Saved 20 test cases to Colab


## Step 3: BM25 Keyword Search Baseline

We used BM25 as our baseline search method to compare against semantic search. BM25 ranks movies based on how closely the words in a description match the words in the movie plot summaries.

We converted the plot summaries to lowercase and split them into words before building the BM25 search index. We then created a function that returns the five highest-ranked movies for each description.

To evaluate BM25, we checked whether the correct movie appeared in the top five results for each answerable test case. We used Hit Rate@5 to measure its performance.

In [5]:
# Install the library for BM25 keyword search
!pip install rank-bm25 -q

In [6]:
# Import the BM25 search algorithm
from rank_bm25 import BM25Okapi

# Convert each plot summary to lowercase and split it into words
tokenized_plots = [plot.lower().split() for plot in df['PlotSummary']]

# Build the BM25 search index using the movie summaries
bm25 = BM25Okapi(tokenized_plots)

# Confirm that all movie summaries were indexed
print("Movies indexed:", len(tokenized_plots))

Movies indexed: 34886


In [7]:
# Search for the five movies that best match a description
def search_bm25(description, top_k=5):
    # Prepare the description the same way as the movie summaries
    query = description.lower().split()

    # Calculate BM25 scores for every movie
    scores = bm25.get_scores(query)

    # Find the five movies with the highest scores
    top_indices = scores.argsort()[-top_k:][::-1]

    # Return the matching movie information
    return df.iloc[top_indices][['Record ID', 'Title', 'Release Year']]

In [8]:
# Store the results for each test description
results = []

# Search for each description in the test set
for _, test in test_df.iterrows():
    matches = search_bm25(test['Description'])

    # Check if the correct movie appears in the top five
    if pd.notna(test['Record_ID']):
        hit = int(test['Record_ID'] in matches['Record ID'].values)
    else:
        hit = None

    # Save the test number, expected movie, and result
    results.append([test['Test'], test['Expected_Title'], hit])

# Display the evaluation results
results_df = pd.DataFrame(results, columns=['Test', 'Expected Movie', 'Hit@5'])
results_df

,Test,Expected Movie,Hit@5
0,1,Jack and the Beanstalk,1.0
1,2,Alice in Wonderland,1.0
2,3,The Truman Show,0.0
3,4,The Playhouse,0.0
4,5,Unanswerable,NaN
5,6,5 Against the House,0.0
6,7,Abbott and Costello Meet the Mummy,0.0
7,8,36 Hours,0.0
8,9,The Matrix,0.0
9,10,The Agony and the Ecstasy,0.0


In [9]:

# Count the answerable test cases and correct matches
answerable_tests = results_df['Hit@5'].count()
correct_matches = results_df['Hit@5'].sum()

# Calculate the BM25 Hit Rate@5
hit_rate = correct_matches / answerable_tests

# Display the baseline results
print("Answerable Tests:", answerable_tests)
print("Correct Matches:", int(correct_matches))
print(f"BM25 Hit Rate@5: {hit_rate:.2%}")


Answerable Tests: 18
Correct Matches: 3
BM25 Hit Rate@5: 16.67%


In [10]:

# Check the top five results for three missed test cases
for test_number in [3, 6, 7]:
    # Get the test description and expected movie
    test = test_df.iloc[test_number - 1]

    print("\nTest:", test_number)
    print("Expected Movie:", test['Expected_Title'])
    print("Description:", test['Description'])

    # Display the five movies returned by BM25
    print(search_bm25(test['Description']).to_string(index=False))



Test: 3
Expected Movie: The Truman Show
Description: The one where a man slowly discovers that his life is being filmed for a secret television show.
 Record ID                  Title  Release Year
      9681            Death Watch          1980
       396          The Lucky Dog          1921
      2778       Diamond Frontier          1940
      7240     The Last Angry Man          1959
      5926 The Marshal's Daughter          1953

Test: 6
Expected Movie: 5 Against the House
Description: A group of college friends visits a casino, and one becomes determined to pull off a robbery while another struggles with trauma from the war.
 Record ID              Title  Release Year
     18320    Crime Unlimited          1935
     12429          Fall Time          1995
      3484    Youth on Parade          1942
      7122 The Space Children          1958
     22048              Steal          2002

Test: 7
Expected Movie: Abbott and Costello Meet the Mummy
Description: Two Americans get stran

### BM25 Baseline Results

We used BM25 to search through 34,886 movie plot summaries using 20 vague movie descriptions. Of the 18 answerable descriptions, BM25 found the correct movie in its top five results for 3 tests, giving it a Hit Rate@5 of 16.67%. The two unanswerable descriptions were excluded from this calculation.

BM25 correctly retrieved Jack and the Beanstalk, Alice in Wonderland, and Little Accidents. It struggled with other descriptions, including The Truman Show, 5 Against the House, and Abbott and Costello Meet the Mummy. This may be because the descriptions use different words than the original plot summaries. We will compare these results with semantic search to see whether it performs better at understanding vague descriptions.

## Step 4: Model Feasibility Testing

We selected two models for our Movie Memory system. We will use thenlper/gte-small to create embeddings for semantic search and Qwen/Qwen2.5-1.5B-Instruct to generate answers using retrieved movie information.

Before building the full system, we want to confirm that both models can load and run in Google Colab.

In [11]:
# Install the library for creating text embeddings
!pip install sentence-transformers -q

In [12]:
# Import the library for creating text embeddings
from sentence_transformers import SentenceTransformer

# Load the embedding model selected for our project
embedding_model = SentenceTransformer('thenlper/gte-small')

# Test the model with a movie description
description = "A man discovers that his life is secretly being filmed."

# Convert the description into an embedding
embedding = embedding_model.encode(description)

# Display the embedding size
print("Embedding dimensions:", len(embedding))

modules.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/68.1k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/583 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 66.7MB            

model.safetensors: downloading bytes:           |  0.00B            

... (3 outputs left out)

tokenizer.json:   0%|          | 0.00/712k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding dimensions: 384


In [13]:
# Import the libraries for loading the language model
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# Set the language model selected for our project
model_name = "Qwen/Qwen2.5-1.5B-Instruct"

# Load the tokenizer and language model
tokenizer = AutoTokenizer.from_pretrained(model_name)
language_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

# Confirm that the model loaded successfully
print("Language model loaded:", model_name)
print("Device:", language_model.device)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Language model loaded: Qwen/Qwen2.5-1.5B-Instruct
Device: cuda:0


In [14]:
# Create a test prompt using a movie plot summary
movie = df[df['Record ID'] == 13380].iloc[0]

prompt = f"""Use the movie information below to identify the movie.

Movie: {movie['Title']}
Plot: {movie['PlotSummary']}

Description: A man discovers that his life is secretly being filmed.

Give the movie title and a short explanation."""

# Format the prompt for the Qwen model
messages = [{"role": "user", "content": prompt}]
inputs = tokenizer.apply_chat_template(
    messages, return_tensors="pt", add_generation_prompt=True
).to(language_model.device)

# Generate a response using the language model
outputs = language_model.generate(**inputs, max_new_tokens=100)

# Display only the generated response
response = tokenizer.decode(
    outputs[0][inputs['input_ids'].shape[1]:],
    skip_special_tokens=True
)
print(response)

The movie title is **"The Truman Show"**. 

**Explanation:** This plot describes a unique premise where the main character, Truman Burbank, lives his entire life under surveillance on a massive set located inside an enormous dome shaped like an oceanic habitat called Seahaven Island. The story revolves around how Truman, unaware of this setup, goes about living his ordinary life without realizing he's actually part of a high-profile reality show that everyone else in the world watches. The film explores themes of


### Model Feasibility Results

We successfully loaded and tested both models in Google Colab. The thenlper/gte-small model generated a 384-dimensional embedding from a movie description.

We also loaded Qwen/Qwen2.5-1.5B-Instruct using a GPU. We tested it by providing a description and the plot summary for The Truman Show. The model correctly identified the movie and generated an explanation based on the provided information. This test checked whether the model could generate an answer using provided movie information, not whether it could find the correct movie on its own.

These tests confirmed that both models can run in our Colab environment.

In [15]:
# Evaluate Qwen without retrieved plots on all 20 test cases
qwen_results = []

for _, test in test_df.iterrows():
    prompt = f"""Identify the movie from this vague description.

Description: {test['Description']}

Give the movie title and a short explanation. If there is not enough information, say you cannot identify the movie."""

    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        messages, return_tensors="pt", add_generation_prompt=True
    ).to(language_model.device)

    outputs = language_model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=False
    )

    response = tokenizer.decode(
        outputs[0][inputs['input_ids'].shape[1]:],
        skip_special_tokens=True
    )

    qwen_results.append([
        test['Test'],
        test['Expected_Title'],
        response
    ])

qwen_results_df = pd.DataFrame(
    qwen_results,
    columns=['Test', 'Expected Movie', 'Qwen Response']
)

qwen_results_df

,Test,Expected Movie,Qwen Response
0,1,Jack and the Beanstalk,I cannot identify the specific movie based on ...
1,2,Alice in Wonderland,The movie described in the vague description i...
2,3,The Truman Show,The movie described in the vague description i...
3,4,The Playhouse,"Based on the vague description provided, it se..."
4,5,Unanswerable,I cannot identify the specific movie based on ...
5,6,5 Against the House,I cannot identify the specific movie based on ...
6,7,Abbott and Costello Meet the Mummy,"Based on the vague description provided, I can..."
7,8,36 Hours,I cannot identify the specific movie based on ...
8,9,The Matrix,"Based on the vague description provided, ""The ..."
9,10,The Agony and the Ecstasy,I cannot identify the specific movie based on ...


In [16]:
# Record manually reviewed Qwen baseline scores
correct_titles = [3, 9, 14]
correct_unanswerable = [5]

qwen_results_df['Correct_Title'] = qwen_results_df['Test'].isin(correct_titles).astype(int)
qwen_results_df['Correct_Abstention'] = qwen_results_df['Test'].isin(correct_unanswerable).astype(int)

print("Qwen Title Accuracy:", f"{len(correct_titles)}/18 = {len(correct_titles)/18:.2%}")
print("Unanswerable Cases:", f"{len(correct_unanswerable)}/2")

Qwen Title Accuracy: 3/18 = 16.67%
Unanswerable Cases: 1/2


### Qwen-Only Baseline Results

We tested Qwen on 20 vague movie descriptions without providing retrieved movie plots. Of the 18 answerable descriptions, Qwen correctly identified 3 movies, giving it a title accuracy of 16.67%. It also correctly recognized 1 of the 2 unanswerable descriptions.

Qwen correctly identified The Truman Show, The Matrix, and Back to the Future. However, some of its explanations contained inaccurate information. For example, Qwen incorrectly stated that Marty's father invented the DeLorean in Back to the Future. It also provided incorrect cast information for Alice in Wonderland.

## Step 5: Semantic Search with FAISS

We use the gte-small embedding model to convert movie plot summaries into numerical vectors. FAISS indexes these vectors so we can retrieve movies based on semantic similarity rather than keyword matches.

In [17]:
# Install FAISS for semantic search
!pip install -q faiss-cpu

In [18]:
# Convert movie plot summaries into embeddings
movie_embeddings = embedding_model.encode(
    df['PlotSummary']. tolist(),
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True
)

print("Embeddings shape:", movie_embeddings.shape)

Batches:   0%|          | 0/546 [00:00<?, ?it/s]

Embeddings shape: (34886, 384)


In [19]:
# Build a FAISS index using cosine similarity
import faiss
import numpy as np

# Convert embeddings to float32 for FAISS
movie_embeddings = np.ascontiguousarray(movie_embeddings, dtype=np.float32)

# Normalize embeddings so inner product measures cosine similarity
faiss.normalize_L2(movie_embeddings)

# Create the index and add all movie embeddings
index = faiss.IndexFlatIP(movie_embeddings.shape[1])
index.add(movie_embeddings)

print("Movies indexed:", index.ntotal)
print("Embedding dimensions:", index.d)

Movies indexed: 34886
Embedding dimensions: 384


In [20]:
# Search for movies using semantic similarity
def search_semantic(description, top_k=5):
    query_embedding = embedding_model.encode(
        [description],
        convert_to_numpy=True
    )

    query_embedding = np.ascontiguousarray(query_embedding, dtype=np.float32)
    faiss.normalize_L2(query_embedding)

    scores, indices = index.search(query_embedding, top_k)

    matches = df.iloc[indices[0]][['Record ID', 'Title', 'Release Year']].copy()
    matches['Similarity Score'] = scores[0]

    return matches


# Test semantic search using an existing test description
test = test_df[test_df['Test'] == 3].iloc[0]

print("Expected Movie:", test['Expected_Title'])
print("Description:", test['Description'])
print(search_semantic(test['Description']).to_string(index=False))

Expected Movie: The Truman Show
Description: The one where a man slowly discovers that his life is being filmed for a secret television show.
 Record ID                Title  Release Year  Similarity Score
      2512            Blackmail          1939          0.882470
     20627            Crimetime          1996          0.869794
     26946 Singh Saab The Great          2013          0.868149
     21827         The Blue Man          1985          0.866032
      1132        The Guardsman          1931          0.865843


In [21]:
# Evaluate semantic search on all test descriptions
semantic_results = []

for _, test in test_df.iterrows():
  matches = search_semantic(test['Description'], top_k=5)

  if pd.notna(test['Record_ID']):
    hit = int(test['Record_ID'] in matches['Record ID'].values)
  else:
    hit = None

  semantic_results.append([
      test['Test'],
      test['Expected_Title'],
      hit
  ])

semantic_results_df = pd.DataFrame(
    semantic_results,
    columns=['Test', 'Expected Movie', 'Hit@5']
)

display(semantic_results_df)

# Calculate Hit Rate@5 for answerable cases
answerable_tests = semantic_results_df['Hit@5'].count()
correct_matches = semantic_results_df['Hit@5'].sum()
semantic_hit_rate = correct_matches / answerable_tests

print("Answerable Tests:", answerable_tests)
print("Correct Matches:", int(correct_matches))
print(f"Semantic Search Hit Rate@5: {semantic_hit_rate:.2%}")

,Test,Expected Movie,Hit@5
0,1,Jack and the Beanstalk,1.0
1,2,Alice in Wonderland,1.0
2,3,The Truman Show,0.0
3,4,The Playhouse,0.0
4,5,Unanswerable,NaN
5,6,5 Against the House,1.0
6,7,Abbott and Costello Meet the Mummy,1.0
7,8,36 Hours,1.0
8,9,The Matrix,0.0
9,10,The Agony and the Ecstasy,1.0


Answerable Tests: 18
Correct Matches: 7
Semantic Search Hit Rate@5: 38.89%


## Step 6: Retrieval-Augmented Generation (RAG)

We combine FAISS semantic search with Qwen to identify movies from vague descriptions. FAISS retrieves relevant movie plot summaries, and Qwen uses those summaries to identify the movie and provide an explanation grounded in the retrieved information.

In [22]:
# Identify a movie using retrieved plots and Qwen
def identify_movie_rag(description, top_k=5):
    matches = search_semantic(description, top_k=top_k)

    # Get the retrieved movie plots
    retrieved_plots = []
    for _, movie in matches.iterrows():
        record = df[df['Record ID'] == movie['Record ID']].iloc[0]
        retrieved_plots.append(
            f"Title: {record['Title']}\n"
            f"Year: {record['Release Year']}\n"
            f"Plot: {record['PlotSummary']}"
        )

    context = "\n\n".join(retrieved_plots)



    # Ask Qwen to select a movie from the retrieved plots
    prompt = f"""You are identifying a movie from a vague description.

    Description:
    {description}

    Candidate movies (ranked by semantic similarity):
    {context}

    Instructions:
    - Compare the description with the actual plot details of each candidate.
    - Select only a movie title from the candidate list.
    - Do not invent a title, year, or plot details.
    - If no candidate has enough supporting evidence, answer Unanswerable.
    - Respond using exactly this format:

    Movie: [exact candidate title or Unanswerable]
    Explanation: [one or two sentences explaining the evidence]
    """



    # Format the prompt for Qwen
    messages = [{"role": "user", "content": prompt}]

    inputs = tokenizer.apply_chat_template(
        messages,
        return_tensors="pt",
        return_dict=True,
        add_generation_prompt=True
    ).to(language_model.device)

    # Generate the response
    outputs = language_model.generate(
      **inputs,
      max_new_tokens=150,
      do_sample=False,
      pad_token_id=tokenizer.eos_token_id
    )

    # Decode only the newly generated tokens
    response = tokenizer.decode(
      outputs[0][inputs["input_ids"].shape[1]:],
      skip_special_tokens=True
    )

        # Check that Qwen selected a movie from the retrieved results
    answer_line = next(
        (line for line in response.splitlines() if line.startswith("Movie:")),
        ""
    )

    selected_title = answer_line.replace("Movie:", "", 1).strip()
    valid_titles = matches['Title'].tolist()

    if selected_title != "Unanswerable" and selected_title not in valid_titles:
        response = "Movie: Unanswerable\nExplanation: The generated title was not in the retrieved results."

    return response, matches

## Step 7: Full RAG Evaluation

We evaluate the complete FAISS and Qwen RAG system using our 20 test descriptions. We compare its movie predictions against the expected titles and review whether its explanations are supported by the retrieved movie plots. We also evaluate the two unanswerable descriptions separately.

In [23]:
# Run the top-5 RAG system on all 20 test descriptions
rag_top5_results = []

for _, test in test_df.iterrows():
    response, matches = identify_movie_rag(test['Description'], top_k=5)

    rag_top5_results.append({
        'Test': test['Test'],
        'Expected Movie': test['Expected_Title'],
        'Qwen RAG Response': response,
        'Retrieved Titles': matches['Title'].tolist(),
        'Expected Record ID': test['Record_ID'],
        'Retrieved Record IDs': matches['Record ID'].tolist()
    })

    print(f"Completed test {test['Test']}")

rag_top5_df = pd.DataFrame(rag_top5_results)

# Display all 20 results
rag_top5_df[['Test', 'Expected Movie', 'Qwen RAG Response']]

Completed test 1
Completed test 2
Completed test 3
Completed test 4
Completed test 5
Completed test 6
Completed test 7
Completed test 8
Completed test 9
Completed test 10
Completed test 11
Completed test 12
Completed test 13
Completed test 14
Completed test 15
Completed test 16
Completed test 17
Completed test 18
Completed test 19
Completed test 20


,Test,Expected Movie,Qwen RAG Response
0,1,Jack and the Beanstalk,Movie: Tweety and the Beanstalk\nExplanation: ...
1,2,Alice in Wonderland,Movie: Alice in Wonderland\nExplanation: This ...
2,3,The Truman Show,Movie: Blackmail\nExplanation: The description...
3,4,The Playhouse,Movie: Unanswerable\nExplanation: None of the ...
4,5,Unanswerable,Movie: The Quarrel\nExplanation: The descripti...
5,6,5 Against the House,Movie: 5 Against the House\nExplanation: This ...
6,7,Abbott and Costello Meet the Mummy,Movie: Abbott and Costello Meet the Mummy\nExp...
7,8,36 Hours,Movie: 36 Hours\nExplanation: The description ...
8,9,The Matrix,Movie: The Lawnmower Man 2: Beyond Cyberspace\...
9,10,The Agony and the Ecstasy,Movie: The Agony and the Ecstasy\nExplanation:...


In [24]:
# Run the RAG system on all 20 test descriptions
rag_top3_results = []

for _, test in test_df.iterrows():
    response, matches = identify_movie_rag(test['Description'], top_k=3)
    rag_top3_results.append({
        'Test': test['Test'],
        'Expected Movie': test['Expected_Title'],
        'Qwen RAG Response': response,
        'Retrieved Titles': matches['Title'].tolist(),
        'Expected Record ID': test['Record_ID'],
        'Retrieved Record IDs': matches['Record ID'].tolist()
    })

    print(f"Completed test {test['Test']}")

rag_top3_df = pd.DataFrame(rag_top3_results)

# Display all 20 results
rag_top3_df[['Test', 'Expected Movie', 'Qwen RAG Response']]

Completed test 1
Completed test 2
Completed test 3
Completed test 4
Completed test 5
Completed test 6
Completed test 7
Completed test 8
Completed test 9
Completed test 10
Completed test 11
Completed test 12
Completed test 13
Completed test 14
Completed test 15
Completed test 16
Completed test 17
Completed test 18
Completed test 19
Completed test 20


,Test,Expected Movie,Qwen RAG Response
0,1,Jack and the Beanstalk,Movie: Jack and the Beanstalk\nExplanation: Th...
1,2,Alice in Wonderland,Movie: Alice in Wonderland\nExplanation: The d...
2,3,The Truman Show,Movie: Blackmail\nExplanation: The description...
3,4,The Playhouse,Movie: Cigarette Ki Tarah\nExplanation: The de...
4,5,Unanswerable,Movie: The Quarrel\nExplanation: The descripti...
5,6,5 Against the House,Movie: 5 Against the House\nExplanation: The d...
6,7,Abbott and Costello Meet the Mummy,Movie: Abbott and Costello Meet the Mummy\nExp...
7,8,36 Hours,Movie: The Devil Pays Off\nExplanation: The de...
8,9,The Matrix,Movie: The Lawnmower Man 2: Beyond Cyberspace\...
9,10,The Agony and the Ecstasy,Movie: The Agony and the Ecstasy\nExplanation:...


### RAG Design Experiment: Top 3 vs. Top 5 Retrieved Plots

We compared two versions of our RAG system using the same 20 movie descriptions, embedding model, Qwen model, and prompt. The only difference was the number of retrieved movie plots provided to Qwen.

- **Top 5:** Correctly identified 6 of 18 answerable movies (33.33%).
- **Top 3:** Correctly identified 6 of 18 answerable movies (33.33%).
- **Unanswerable cases:** Neither version correctly rejected the two descriptions without enough information.

Both settings achieved the same overall accuracy, although they correctly identified different movies. This suggests that reducing the number of retrieved plots did not improve overall identification accuracy on our test set. Our results also show that Qwen sometimes selects an incorrect movie even when the correct movie appears among the retrieved results.

In [25]:
# Score the final movie identification results
def score_rag_results(results):
    scored = results.copy()

    # Extract the movie title from Qwen's response
    scored['Predicted Movie'] = scored['Qwen RAG Response'].str.extract(
        r'Movie:\s*([^\n]+)'
    )[0].str.strip()

    # Compare predicted titles with expected titles
    scored['Correct'] = (
        scored['Predicted Movie'].str.casefold()
        == scored['Expected Movie'].str.casefold()
    )

    answerable = scored['Expected Movie'] != 'Unanswerable'
    correct = int(scored.loc[answerable, 'Correct'].sum())
    total = int(answerable.sum())

    unanswerable_correct = int(scored.loc[~answerable, 'Correct'].sum())

    print(f"Correct answerable movies: {correct}/{total}")
    print(f"RAG accuracy: {correct / total:.2%}")
    print(f"Correct unanswerable cases: {unanswerable_correct}/2")

    return scored

print("TOP 5 RESULTS")
top5_scored = score_rag_results(rag_top5_df)

print("\nTOP 3 RESULTS")
top3_scored = score_rag_results(rag_top3_df)

TOP 5 RESULTS
Correct answerable movies: 6/18
RAG accuracy: 33.33%
Correct unanswerable cases: 0/2

TOP 3 RESULTS
Correct answerable movies: 6/18
RAG accuracy: 33.33%
Correct unanswerable cases: 0/2


## Final Evaluation Results

This section combines the evaluation results for all five methods using the same test set of 20 movie descriptions. Hit Rate@5 measures retrieval performance, while title accuracy measures movie identification performance. Both metrics are calculated using the 18 answerable test cases.

- **Grounding Rate:** Percentage of non-abstaining responses with factually supported explanations, based on manual evaluation.
- **Correct Abstention Rate:** Percentage of the two unanswerable descriptions that the model correctly declined to identify.

In [32]:
# Record manually evaluated RAG Top 5 grounding scores
top5_scored['Grounded'] = [
    1, 1, 0, None, 0,
    0, 1, 0, 0, 1,
    1, 1, 1, 0, 1,
    1, 0, 0, 0, 1
]

print(f"Grounded explanations: {top5_scored['Grounded'].sum():.0f}/{top5_scored['Grounded'].count()}")
print(f"Grounding rate: {top5_scored['Grounded'].mean():.2%}")

Grounded explanations: 10/19
Grounding rate: 52.63%


In [33]:
# Record manually evaluated RAG Top 3 grounding scores
top3_scored['Grounded'] = [
    1, 1, 1, 0, 0,
    1, 1, 1, 0, 1,
    0, 0, 0, 1, 1,
    0, 1, 1, 0, 0
]

print(f"Grounded explanations: {top3_scored['Grounded'].sum():.0f}/{top3_scored['Grounded'].count()}")
print(f"Grounding rate: {top3_scored['Grounded'].mean():.2%}")

Grounded explanations: 11/20
Grounding rate: 55.00%


In [34]:
# Record manually evaluated Qwen-only grounding scores
qwen_results_df['Grounded'] = [
    None, 0, 0, 0, None,
    None, 0, None, 1, None,
    None, None, 0, 0, 0,
    None, None, None, 0, 0
]

print(f"Grounded explanations: {qwen_results_df['Grounded'].sum():.0f}/{qwen_results_df['Grounded'].count()}")
print(f"Grounding rate: {qwen_results_df['Grounded'].mean():.2%}")

Grounded explanations: 1/10
Grounding rate: 10.00%


In [35]:
# Combine final evaluation results
final_results_df = pd.DataFrame([
    ['BM25 Keyword Search', 'Hit Rate@5', results_df['Hit@5'].mean()],
    ['FAISS Semantic Search', 'Hit Rate@5', semantic_results_df['Hit@5'].mean()],
    ['Qwen-Only Baseline', 'Title Accuracy', qwen_results_df['Correct_Title'].sum() / answerable_tests],
    ['RAG Top 5', 'Title Accuracy', top5_scored.loc[top5_scored['Expected Movie'] != 'Unanswerable', 'Correct'].mean()],
    ['RAG Top 3', 'Title Accuracy', top3_scored.loc[top3_scored['Expected Movie'] != 'Unanswerable', 'Correct'].mean()],
    ['Qwen-Only Baseline', 'Grounding Rate', qwen_results_df['Grounded'].mean()],
    ['RAG Top 5', 'Grounding Rate', top5_scored['Grounded'].mean()],
    ['RAG Top 3', 'Grounding Rate', top3_scored['Grounded'].mean()],
    ['Qwen-Only Baseline', 'Correct Abstention Rate', qwen_results_df.loc[qwen_results_df['Expected Movie'] == 'Unanswerable', 'Correct_Abstention'].mean()],
    ['RAG Top 5', 'Correct Abstention Rate', top5_scored.loc[top5_scored['Expected Movie'] == 'Unanswerable', 'Correct'].mean()],
    ['RAG Top 3', 'Correct Abstention Rate', top3_scored.loc[top3_scored['Expected Movie'] == 'Unanswerable', 'Correct'].mean()]
], columns=['Method', 'Metric', 'Score'])

# Format and display results
final_results_df['Score'] = final_results_df['Score'].map('{:.2%}'.format)
display(final_results_df)

# Save final evaluation results
final_results_df.to_csv('movie_memory_final_results.csv', index=False)

,Method,Metric,Score
0,BM25 Keyword Search,Hit Rate@5,16.67%
1,FAISS Semantic Search,Hit Rate@5,38.89%
2,Qwen-Only Baseline,Title Accuracy,16.67%
3,RAG Top 5,Title Accuracy,33.33%
4,RAG Top 3,Title Accuracy,33.33%
5,Qwen-Only Baseline,Grounding Rate,10.00%
6,RAG Top 5,Grounding Rate,52.63%
7,RAG Top 3,Grounding Rate,55.00%
8,Qwen-Only Baseline,Correct Abstention Rate,50.00%
9,RAG Top 5,Correct Abstention Rate,0.00%


## Failure Analysis

### Failure 1 — Incorrect Movie Identification
- Test: 3 (RAG Top-5)
- Expected movie: The Truman Show
- Predicted movie: Blackmail
- Expected Record ID: 13380
- Retrieved Record IDs: 2512, 20627, 26946, 21827, 1132
- What went wrong: The correct movie was not among the five retrieved candidates, and Qwen incorrectly selected Blackmail.
- Potential improvement: Use a more capable embedding model.

### Failure 2 — Incorrect Explanation
- Test: 14 (Qwen-Only Baseline)
- Expected movie: Back to the Future
- Predicted movie: Back to the Future
- What went wrong: Qwen incorrectly stated that Marty's father invented the time machine, while the reference plot identifies Dr. Emmett Brown as the inventor.
- Potential improvement: Require explanations to be supported by retrieved information.

In [36]:
# Save individual evaluation results
results_df.to_csv('bm25_results.csv', index=False)
semantic_results_df.to_csv('semantic_results.csv', index=False)
qwen_results_df.to_csv('qwen_results.csv', index=False)
top5_scored.to_csv('rag_top5_results.csv', index=False)
top3_scored.to_csv('rag_top3_results.csv', index=False)
test_df.to_csv('movie_memory_20_test_cases.csv', index=False)

In [37]:
# Record package versions for reproducibility
from importlib.metadata import version

packages = [
    'pandas',
    'numpy',
    'torch',
    'transformers',
    'sentence-transformers',
    'faiss-cpu',
    'rank-bm25',
    'gdown'
]

for package in packages:
    print(f"{package}: {version(package)}")

pandas: 2.2.3
numpy: 2.4.6
torch: 2.14.1+cu130
transformers: 5.22.1
sentence-transformers: 5.7.0
faiss-cpu: 1.16.0
rank-bm25: 0.2.2
gdown: 5.2.2
